In [3]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import warnings

# Ignorar advertencias de geopandas
warnings.filterwarnings("ignore")

def diagnosticar_capa(ruta_archivo):
    # Diccionario para almacenar el resultado inicial
    diagnostico = {
        "Nombre": ruta_archivo.name,
        "Ruta": str(ruta_archivo),
        "Estado": "Pendiente",
        "Registros": 0,
        "Columnas": 0,
        "Nombres_Columnas": "",
        "Tipos_Datos": "",
        "CRS": "",
        "Tipo_Geometria": "",
        "X_min": None,
        "Y_min": None,
        "X_max": None,
        "Y_max": None,
        "Geometrias_Nulas": 0,
        "Geometrias_Validas": 0,
        "Geometrias_Invalidas": 0
    }
    
    # Verificar existencia del archivo
    if not ruta_archivo.exists():
        diagnostico["Estado"] = "Error: Archivo inexistente"
        return diagnostico, None
        
    try:
        # Cargar la capa con GeoPandas
        gdf = gpd.read_file(ruta_archivo)
        
        # Gestionar capa vacia
        if gdf.empty or gdf.geometry.isnull().all():
            diagnostico["Estado"] = "Error: Capa vacia o sin geometrias"
            return diagnostico, None
            
        # Extraer informacion
        diagnostico["Estado"] = "Lectura exitosa"
        diagnostico["Registros"] = len(gdf)
        diagnostico["Columnas"] = len(gdf.columns)
        diagnostico["Nombres_Columnas"] = ", ".join(gdf.columns.tolist())
        diagnostico["Tipos_Datos"] = ", ".join([str(dtype) for dtype in gdf.dtypes])
        diagnostico["CRS"] = str(gdf.crs)
        
        # Geometrias
        tipos_geom = gdf.geometry.geom_type.unique()
        diagnostico["Tipo_Geometria"] = ", ".join([str(t) for t in tipos_geom])
        
        # Validacion
        diagnostico["Geometrias_Nulas"] = int(gdf.geometry.isnull().sum())
        geometrias_validas = gdf.geometry.is_valid
        diagnostico["Geometrias_Validas"] = int(geometrias_validas.sum())
        diagnostico["Geometrias_Invalidas"] = int((~geometrias_validas & ~gdf.geometry.isnull()).sum())
        
        # Extension
        bounds = gdf.total_bounds
        diagnostico["X_min"] = bounds[0]
        diagnostico["Y_min"] = bounds[1]
        diagnostico["X_max"] = bounds[2]
        diagnostico["Y_max"] = bounds[3]
        
        head_df = gdf.head(5)
        return diagnostico, head_df
        
    except Exception as e:
        diagnostico["Estado"] = f"Error de lectura: {str(e)}"
        return diagnostico, None

def main():
    # NUEVA RUTA DE ENTRADA CON TUS CORTES
    carpeta_datos = Path(r"C:\Users\Jorge Tirado\Documents\Representación de información geoespacial en la web\Cortes de culiacan")
    
    # Ruta de salida original
    ruta_salida = Path(r"C:\Users\Jorge Tirado\Documents\Representación de información geoespacial en la web\Salida\01_diagnostico_capas.csv")
    ruta_salida.parent.mkdir(parents=True, exist_ok=True)
    
    # LECTURA DINAMICA DE ARCHIVOS
    # Busca todos los archivos con extension .shp dentro de la carpeta_datos
    archivos = list(carpeta_datos.glob("*.shp"))
    
    # Trampa de seguridad por si la carpeta esta vacia
    if not archivos:
        print(f"Error fatal: No se encontro ningun archivo .shp en la carpeta:")
        print(f"{carpeta_datos}")
        print("Asegurate de haber exportado tus cortes de QGIS correctamente.")
        return
    
    resultados_consolidados = []
    capas_procesadas = 0
    capas_con_errores = 0
    
    print("Iniciando exploracion dinamica...\n" + "-"*50)
    
    # Ahora iteramos sobre los objetos Path encontrados, no sobre cadenas de texto
    for ruta_completa in archivos:
        nombre_archivo = ruta_completa.name
        print(f"\nProcesando: {nombre_archivo}")
        diagnostico, head_df = diagnosticar_capa(ruta_completa)
        
        for clave, valor in diagnostico.items():
            print(f"{clave.replace('_', ' ')}: {valor}")
            
        if head_df is not None:
            print("\nPrimeras 5 filas:")
            print(head_df.drop(columns=['geometry'], errors='ignore').to_string())
        
        print("-" * 50)
        resultados_consolidados.append(diagnostico)
        
        if "Error" in diagnostico["Estado"]:
            capas_con_errores += 1
        else:
            capas_procesadas += 1
            
    df_resultados = pd.DataFrame(resultados_consolidados)
    
    print("\nRESUMEN CONSOLIDADO:")
    print(df_resultados[["Nombre", "Estado", "Registros", "CRS"]].to_string())
    
    try:
        df_resultados.to_csv(ruta_salida, index=False, encoding='utf-8')
        estado_guardado = f"Guardado exitosamente en: {ruta_salida}"
    except Exception as e:
        estado_guardado = f"Error al guardar: {str(e)}"
        
    print("\n" + "="*50)
    print("REPORTE FINAL")
    print(f"Total de capas encontradas: {len(archivos)}")
    print(f"Capas procesadas correctamente: {capas_procesadas}")
    print(f"Capas con errores: {capas_con_errores}")
    print(estado_guardado)
    print("="*50)

if __name__ == "__main__":
    main()

Iniciando exploracion dinamica...
--------------------------------------------------

Procesando: corte.shp
Nombre: corte.shp
Ruta: C:\Users\Jorge Tirado\Documents\Representación de información geoespacial en la web\Cortes de culiacan\corte.shp
Estado: Error: Capa vacia o sin geometrias
Registros: 0
Columnas: 0
Nombres Columnas: 
Tipos Datos: 
CRS: 
Tipo Geometria: 
X min: None
Y min: None
X max: None
Y max: None
Geometrias Nulas: 0
Geometrias Validas: 0
Geometrias Invalidas: 0
--------------------------------------------------

Procesando: corteclima.shp
Nombre: corteclima.shp
Ruta: C:\Users\Jorge Tirado\Documents\Representación de información geoespacial en la web\Cortes de culiacan\corteclima.shp
Estado: Lectura exitosa
Registros: 8
Columnas: 13
Nombres Columnas: AREA, PERIMETER, COV_, COV_ID, AP1, CLIMA_LLAV, CLIMA_SC, CLIMA_TP, CLIMA_SF, CLIMA_TIPO, DES_TEM, DESC_PREC, geometry
Tipos Datos: float64, float64, int64, int64, int32, str, int32, int32, int32, str, str, str, geometry
CR

# Explorador Topológico de Capas Vectoriales 🗺️🐍

Este es un script de Python diseñado para leer dinámicamente directorios llenos de archivos Shapefile (`.shp`) y extraer sus metadatos espaciales. Básicamente, hace el trabajo sucio de auditoría topológica que los sistemas de información geográfica a veces prefieren ocultar, asegurando que tus recortes vectoriales sean válidos antes de inyectarlos en modelos espaciales.

## 🛠️ Características Principales

*   **Lectura Dinámica (`glob`):** Escanea automáticamente la carpeta de entrada buscando cualquier archivo `.shp`. Si agregas 100 recortes nuevos mañana, el script los procesará todos sin que tengas que modificar ni una sola línea de código.
*   **Tolerancia a Fallos (`try-except`):** Si un archivo está corrupto, mal exportado o nació vacío (un clásico), el script lo documenta y continúa con el siguiente sin provocar que toda la ejecución colapse dramáticamente.
*   **Análisis Geométrico:** Verifica el Sistema de Referencia de Coordenadas (CRS), calcula la extensión espacial (Bounding Box) y contabiliza las geometrías válidas, inválidas y nulas.
*   **Reporte Consolidado:** Exporta todo el diagnóstico a un archivo CSV estructurado, ideal para su revisión en Excel o para integrarlo como log de validación en pipelines de datos.

## 📦 Requisitos del Sistema

Necesitas tener instalados los siguientes paquetes en tu entorno virtual de Python. Se recomienda encarecidamente utilizar `conda` en Windows para evitar los típicos errores de compilación de las dependencias geoespaciales escritas en C.

*   `pandas`
*   `geopandas`

**Comando de instalación (vía Anaconda):**
```bash
conda install -c conda-forge geopandas pandas